In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import timedelta

# =========================================================
# LOAD DATA
# =========================================================

df_suppliers = pd.read_csv("data/raw/df_suppliers.csv")
df_po = pd.read_csv("data/raw/df_po.csv")


# =========================================================
# DELAY REASONS
# =========================================================

delay_reasons = [

    "Port Congestion",
    "Customs Clearance",
    "Weather Disruption",
    "Supplier Production Delay",
    "Documentation Issue",
    "Carrier Capacity Shortage",
    "Labor Strike"

]


# =========================================================
# TRANSPORT MODE RULES
# =========================================================

transport_rules = {

    "electronics": {
        "modes": ["air", "sea"],
        "weights": [0.3, 0.7]
    },

    "metals": {
        "modes": ["sea", "rail"],
        "weights": [0.8, 0.2]
    },

    "packaging": {
        "modes": ["road"],
        "weights": [1.0]
    },

    "chemicals": {
        "modes": ["road", "sea"],
        "weights": [0.6, 0.4]
    },

    "plastics": {
        "modes": ["road", "sea"],
        "weights": [0.7, 0.3]
    },

    "textiles": {
        "modes": ["sea", "air"],
        "weights": [0.8, 0.2]
    }

}


# =========================================================
# SHIPMENT GENERATION
# =========================================================

shipment_rows = []

shipment_counter = 1


for _, po in df_po.iterrows():

    # =====================================================
    # GET SUPPLIER
    # =====================================================

    supplier = df_suppliers[

        df_suppliers["supplier_id"]
        == po["supplier_id"]

    ].iloc[0]


    supplier_id = supplier["supplier_id"]

    category = supplier["supplier_category"]

    country = supplier["country"]

    criticality = supplier["criticality_level"]


    # =====================================================
    # DISPATCH DATE
    # =====================================================
    # Manufacturing/preparation delay
    # =====================================================

    prep_days = {

        "electronics": np.random.randint(7, 25),

        "metals": np.random.randint(5, 18),

        "packaging": np.random.randint(2, 7),

        "chemicals": np.random.randint(5, 15)

    }.get(category, np.random.randint(3, 12))


    dispatch_date = (

        pd.to_datetime(po["order_date"])
        + timedelta(days=int(prep_days))

    )


    # =====================================================
    # TRANSPORT MODE
    # =====================================================

    if category in transport_rules:

        transport_mode = np.random.choice(

            transport_rules[category]["modes"],

            p=transport_rules[category]["weights"]

        )

    else:

        transport_mode = "road"


    # =====================================================
    # INTERNATIONAL VS DOMESTIC
    # =====================================================

    international = country != "India"


    # =====================================================
    # TRANSIT TIME
    # =====================================================

    transit_days = {

        "air": np.random.randint(2, 10),

        "sea": np.random.randint(20, 60),

        "road": np.random.randint(2, 12),

        "rail": np.random.randint(7, 20)

    }[transport_mode]


    # Domestic shipments are faster
    if not international:

        transit_days *= 0.6


    # High criticality suppliers
    # often receive expedited handling
    if criticality >= 4:

        transit_days *= 0.85


    transit_days = max(1, int(transit_days))


    # =====================================================
    # ETA
    # =====================================================

    eta = (

        dispatch_date
        + timedelta(days=transit_days)

    )


    # =====================================================
    # SHIPMENT STATUS
    # =====================================================

    status = np.random.choice(

        [
            "Delivered",
            "Delayed",
            "In Transit",
            "Cancelled"
        ],

        p=[0.70, 0.20, 0.07, 0.03]

    )


    # =====================================================
    # ARRIVAL + DELAYS
    # =====================================================

    actual_arrival = None

    delay_reason = None


    if status == "Delivered":

        # slight early/on-time variability
        arrival_variance = np.random.randint(-2, 3)

        actual_arrival = (

            eta
            + timedelta(days=arrival_variance)

        )


    elif status == "Delayed":

        delay_days = np.random.randint(2, 15)

        actual_arrival = (

            eta
            + timedelta(days=delay_days)

        )

        delay_reason = random.choice(
            delay_reasons
        )


    elif status == "In Transit":

        actual_arrival = None


    elif status == "Cancelled":

        actual_arrival = None

        delay_reason = "Shipment Cancelled"


    # =====================================================
    # CREATE SHIPMENT RECORD
    # =====================================================

    shipment_rows.append({

        "shipment_id":
            f"SHP{shipment_counter:07}",

        "po_id":
            po["po_id"],

        "supplier_id":
            supplier_id,

        "dispatch_date":
            dispatch_date.date(),

        "eta":
            eta.date(),

        "actual_arrival":
            (
                actual_arrival.date()
                if actual_arrival is not None
                else None
            ),

        "transport_mode":
            transport_mode,

        "shipment_status":
            status,

        "delay_reason":
            delay_reason
    })


    shipment_counter += 1


# =========================================================
# FINAL DATAFRAME
# =========================================================

df_shipments = pd.DataFrame(
    shipment_rows
)


# =========================================================
# OPTIONAL VALIDATION
# =========================================================

print("\n==============================")
print("SHIPMENT SUMMARY")
print("==============================")

print(
    f"Total Shipments: "
    f"{len(df_shipments)}"
)

print(
    "\nShipment Status Distribution:"
)

print(
    df_shipments[
        "shipment_status"
    ].value_counts()
)

print(
    "\nTransport Mode Distribution:"
)

print(
    df_shipments[
        "transport_mode"
    ].value_counts()
)

print("\nSample Records:\n")

print(
    df_shipments.head()
)

In [ ]:
df_shipments.info()

In [ ]:
df_shipments.head(5)

In [ ]:
df_shipments.columns

In [ ]:
mask = (
    pd.to_datetime(df_shipments["actual_arrival"], errors="coerce")
    < pd.to_datetime(df_shipments["dispatch_date"], errors="coerce")
)

df_shipments.loc[mask, "actual_arrival"] = df_shipments.loc[mask, "eta"]

mask.sum()

In [ ]:
df_shipments.to_csv('data/raw/df_shipments.csv', index=False)

In [ ]:
# Filter shipments where actual_arrival is NaN
shipments_with_nan_arrival = df_shipments[df_shipments['actual_arrival'].isna()]

# Print unique values of shipment_status for these shipments
print("Unique shipment statuses for shipments with NaN actual_arrival:")
print(shipments_with_nan_arrival['shipment_status'].unique())